In [23]:
"""
Colab & HF setup
- mount GDrive (store your access tokens there if ever)
- authenticate GitHub
- authenticate HuggingFace

If you're not on Colab, you can skip this.
If you're on Colab, run this once per runtime.
"""

!git clone https://github.com/Pacozabala/CSCI199.X-workbench
%cd CSCI199.X-workbench

from google.colab import drive
drive.mount('/content/drive')

with open("/content/drive/MyDrive/keys/ghpat-colab.txt") as f:
  GH_TOKEN = f.read().strip()

import os, subprocess
REPO_URL = "https://github.com/Pacozabala/CSCI199.X-workbench"
GH_USER = os.environ["GH_USER"]

auth_url = REPO_URL.replace(
    "https://",
    f"https://{GH_USER}:{GH_TOKEN}@"
)
subprocess.run(["git", "remote", "set-url", "origin", auth_url], check=True)


with open("/content/drive/MyDrive/keys/hfpat-colab.txt") as f:
  HF_TOKEN = f.read().strip()

from huggingface_hub import login
login(token=HF_TOKEN)

Cloning into 'CSCI199.X-workbench'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 164 (delta 93), reused 117 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 5.60 MiB | 14.15 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/CSCI199.X-workbench/CSCI199.X-workbench/CSCI199.X-workbench/CSCI199.X-workbench/CSCI199.X-workbench
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Flat-classification with MoralBERT

MoralBERT is specified as such:
- ADAM optimizer is used with a learning rate of $5e^{-5}$
- there are $16$ batch sizes and $5$ epochs.

In this notebook MoralBERT is fine-tuned on our augmented MFRC. The lexically polarity-annotated MFRC is one hot encoded with respect to the five moral foundations. After this MoralBERT is bulk loaded and fine-tuned.

In [24]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/datasets/MFRC_polarity_pacoscript.csv")
df = df.dropna(subset=["polarity"])
df.head()

,text,subreddit,bucket,annotator,annotation,confidence,polarity
4,The Le Pen brand of conservatism and classical...,europe,French politics,annotator03,Authority,Somewhat Confident,authority.virtue
12,You are simplifying it. Islam is not the sole ...,europe,French politics,annotator03,Harm,Confident,harm.vice
13,You are simplifying it. Islam is not the sole ...,europe,French politics,annotator01,"Ingroup,Harm",Confident,"ingroup.vice,harm.vice"
14,You are simplifying it. Islam is not the sole ...,europe,French politics,annotator02,Harm,Confident,harm.vice
19,&gt; Valls is such a disgusting traitor to his...,europe,French politics,annotator04,"Ingroup,Purity,Authority",Confident,purity.vice


In [25]:
mfrc_ohe = df[["text", "polarity"]].copy()
mfrc_ohe.head()

,text,polarity
4,The Le Pen brand of conservatism and classical...,authority.virtue
12,You are simplifying it. Islam is not the sole ...,harm.vice
13,You are simplifying it. Islam is not the sole ...,"ingroup.vice,harm.vice"
14,You are simplifying it. Islam is not the sole ...,harm.vice
19,&gt; Valls is such a disgusting traitor to his...,purity.vice


In [26]:
labels = [
    "harm.virtue",
     "harm.vice",
     "authority.virtue",
     "authority.vice",
     "fairness.virtue",
     "fairness.vice",
     "loyalty.virtue",
     "loyalty.vice",
     "purity.virtue",
     "purity.vice"
     ]

for label in labels:
  mfrc_ohe[label] = 0


mfrc_ohe['polarity'] = mfrc_ohe['polarity'].str.replace('ingroup', 'loyalty', regex=False)
mfrc_ohe.head()

,text,polarity,harm.virtue,harm.vice,authority.virtue,authority.vice,fairness.virtue,fairness.vice,loyalty.virtue,loyalty.vice,purity.virtue,purity.vice
4,The Le Pen brand of conservatism and classical...,authority.virtue,0,0,0,0,0,0,0,0,0,0
12,You are simplifying it. Islam is not the sole ...,harm.vice,0,0,0,0,0,0,0,0,0,0
13,You are simplifying it. Islam is not the sole ...,"loyalty.vice,harm.vice",0,0,0,0,0,0,0,0,0,0
14,You are simplifying it. Islam is not the sole ...,harm.vice,0,0,0,0,0,0,0,0,0,0
19,&gt; Valls is such a disgusting traitor to his...,purity.vice,0,0,0,0,0,0,0,0,0,0


In [27]:
dummies = mfrc_ohe['polarity'].str.get_dummies(sep=',')
mfrc_ohe.update(dummies)
mfrc_ohe.head()

,text,polarity,harm.virtue,harm.vice,authority.virtue,authority.vice,fairness.virtue,fairness.vice,loyalty.virtue,loyalty.vice,purity.virtue,purity.vice
4,The Le Pen brand of conservatism and classical...,authority.virtue,0,0,1,0,0,0,0,0,0,0
12,You are simplifying it. Islam is not the sole ...,harm.vice,0,1,0,0,0,0,0,0,0,0
13,You are simplifying it. Islam is not the sole ...,"loyalty.vice,harm.vice",0,1,0,0,0,0,0,1,0,0
14,You are simplifying it. Islam is not the sole ...,harm.vice,0,1,0,0,0,0,0,0,0,0
19,&gt; Valls is such a disgusting traitor to his...,purity.vice,0,0,0,0,0,0,0,0,0,1


In [44]:
!cp /content/drive/MyDrive/Colab\ Notebooks/single-label-classifiers.ipynb .
!git add

commit 535687de68bc581897910e6b95f07d7562fe4c27 (HEAD -> max/single-label-classifiers-approach, origin/max/single-label-classifiers-approach)
Author: jeanmaxcacacho <maxcacacho@gmail.com>
Date:   Sat Feb 28 01:06:04 2026 +0000

    chore: include github authentication in colab setup cell

commit 9e6ca37faf84f811f361bd09fc386a8b70dc35e5
Author: jeanmaxcacacho <maxcacacho@gmail.com>
Date:   Fri Feb 27 08:36:58 2026 +0000

    chore: colab and HF setup in notebook, 'load' roberta-base

commit 9ca06f917289f7ad494b970030d694ab39179d56
Author: jeanmaxcacacho <maxcacacho@gmail.com>
Date:   Fri Feb 27 07:57:08 2026 +0000

    feat: init single label classifiers notebook

commit 01bb2958d747f051284c8faced142f8b13bac87d
Author: Paco Antonio Zabala <157790408+Pacozabala@users.noreply.github.com>
Date:   Fri Feb 27 15:26:47 2026 +0800

    3 epochs training

commit 2160023dc4ec43e50d65707d534b18c00c08fee0
Author: Paco Antonio Zabala <paco.zabala@student.ateneo.edu>
Date:   Thu Feb 26 18:30:16 2026